# DataPilot AI — Evaluation results (no GPU rerun)

**What:** Load **executed** artefacts for training, retrieval (exp_05), and LLM systems A/B/C (exp_01, exp_02).

**Why:** The research question compares a generic LLM with domain RAG ± LoRA. This notebook interprets those runs. It does **not** call `run_experiments.py` (that is `05_experiments.ipynb` on a T4).

**What the numbers mean:** Automatic scores are **lexical proxies** (point coverage, token F1, ROUGE-L), not human 1–5 grades. Inference was **fp16** on Tesla T4 after bitsandbytes 4-bit failed. Do not cite folders whose answers start with `Generation failed`.

Canonical markdown: `experiments/results/tables/results.md`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass

ROOT = Path.cwd()
if not (ROOT / "config" / "evaluation.yaml").exists() and (ROOT.parent / "config" / "evaluation.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT.resolve())

## 2. Held-out Dataset C

**What:** 100 questions isolated from Dataset B (fine-tuning).

**Why:** Fair comparison of Systems A/B/C. Training on eval questions would invalidate the thesis tables.

**Meaning:** Mix of SQL, DE, DW, BI, Analytics, plus 10 out-of-domain items for refusal behaviour.

In [ ]:
import pandas as pd
from src.evaluation.schema import load_eval_jsonl

items = load_eval_jsonl(ROOT / "data" / "evaluation" / "eval_set.jsonl")
edf = pd.DataFrame(items)
print("n:", len(edf))
display(edf["category"].value_counts().rename("n").to_frame())
display(edf["difficulty"].value_counts().rename("n").to_frame())

## 3. Canonical tables (from artefacts)

`collect_results()` skips mock runs and bitsandbytes `generation_error` folders. If Tables 3–4 still show placeholders, the Colab output folders are not in `experiments/results/evaluation/`.

In [ ]:
from IPython.display import Markdown, display
from src.evaluation.tables import collect_results, render_markdown

bundle = collect_results()
print("has_training", bundle.get("training") is not None)
print("has_retrieval", bundle.get("retrieval") is not None)
print("has_exp_01", bundle.get("exp_01") is not None, bundle.get("exp_01_dir"))
print("has_exp_02", bundle.get("exp_02") is not None, bundle.get("exp_02_dir"))
display(Markdown(render_markdown(bundle)))

## 4. How to read the LLM metrics

| Metric | What it measures | Caveat |
|--------|------------------|--------|
| Point coverage | Fraction of expected-answer **points** with lexical overlap in the reply | Misses paraphrases; rewards keyword overlap |
| Token F1 / ROUGE-L | Overlap with the written `reference_answer` | Sensitive to phrasing; FT models that copy instruction style may score higher |
| Latency | Wall-clock `ask()` time | Includes retrieve + generate; first-batch warmup |
| Groundedness proxy | Overlap of the answer with **retrieved** chunks | Only defined when RAG is on |

**Executed pattern (n=100, fp16 T4):**

- RAG vs base LLM: slightly **higher point coverage** (0.51 vs 0.47), **slower**.
- Fine-tuned + RAG vs RAG: **lower** point coverage (0.43 vs 0.51) but **higher** token F1 / ROUGE-L (0.33 / 0.28 vs ~0.13 / 0.11) and **lower** latency.

That is a real trade-off for the discussion chapter: LoRA answers look more like the reference wording, while covering fewer of the checklist points. Do **not** claim human correctness from these numbers.

## 5. Sample predictions (same question, three systems)

Compare `eval_sql_001` across the canonical run folders. Answers must be English SQL/BI, not pip errors.

In [ ]:
from src.evaluation.tables import latest_valid_eval

eval_root = ROOT / "experiments" / "results" / "evaluation"
exp01, _ = latest_valid_eval(eval_root, "exp_01_baseline_vs_rag")
exp02, _ = latest_valid_eval(eval_root, "exp_02_rag_vs_finetuned_rag")

def first_matching(jsonl: Path, qid="eval_sql_001"):
    with jsonl.open(encoding="utf-8") as fh:
        for line in fh:
            row = json.loads(line)
            if row.get("question_id") == qid:
                return row
    return None

qid = "eval_sql_001"
pairs = []
if exp01:
    pairs += [(exp01 / "baseline_llm_predictions.jsonl", "baseline_llm"), (exp01 / "rag_only_predictions.jsonl", "rag_only (exp_01)")]
if exp02:
    pairs += [(exp02 / "finetuned_rag_predictions.jsonl", "finetuned_rag")]

for path, label in pairs:
    row = first_matching(path, qid)
    print("===", label, "===")
    if not row:
        print("missing")
        continue
    print("mode:", row.get("mode"))
    print("point_coverage:", (row.get("metrics") or {}).get("point_coverage"))
    print((row.get("answer") or "")[:500])
    print()

## 6. Retrieval study (no LLM)

Top-k lexical coverage of expected points in FAISS hits. Analytics is weakest; that matches some generation-quality limits later. OOD retrieval mean is low by design of the proxy (OOD items have medical/legal points, not BI jargon).

In [ ]:
from IPython.display import Image, display

fig_dir = ROOT / "experiments" / "results" / "tables" / "figures"
for name in [
    "fig_training_loss.png",
    "fig_eval_loss_epochs.png",
    "fig_retrieval_topk.png",
    "fig_retrieval_by_category.png",
    "fig_retrieval_by_difficulty.png",
]:
    p = fig_dir / name
    print(name, "exists" if p.exists() else "MISSING")
    if p.exists():
        display(Image(filename=str(p)))

## 7. Limitations (cite these, do not hide them)

- Keyword OOD filter refused **5/100** items; Dataset C has **10** OOD questions.
- Optional exp_03 (embeddings), exp_04 (chunk size), exp_06 (latency-only study) were not required and were not run.
- Human 1–5 rubric is a **blank** CSV only (`experiments/results/human_eval/rubric_blank.csv`).
- V1 model is Qwen2.5-1.5B-Instruct + LoRA-fp16; not a 7B production assistant.

GPU re-run (only if artefacts are missing): `notebooks/05_experiments.ipynb` with `faiss-cpu` and `--no-4bit`.